# Hamiltonian Spectrum & Separation Diagnostics (fixed ε = 2 mrad, γ = 3)

Remake of the spectral-diagnostics study on the **new fixed-ε data**.  The
matrix A is never stored — it is regenerated from each stored event
(`qp.build_hamiltonian`, step kernel, γ = 3, δ = 1) and the **stored** solution
vector supplies the true/false split.  Questions:

- **(a)** Do the eigenvalue extremes / condition number κ = λ_max/λ_min stay
  benign as multiplicity grows, or does the spectrum explode?
- **(b)** Is the Gershgorin lower bound min_i(d_i − r_i) informative, or does
  sparsity keep λ_min positive past where the bound hits 0?
- **(c)** Does the true-vs-false solution separation stay clean, or does the
  false tail cross the threshold as n grows (the source of the false-positive
  rate in the efficiency figure)?

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seg_store as S

plt.rcParams.update({"figure.dpi": 110, "font.size": 11,
                      "axes.grid": True, "grid.alpha": 0.3})
print("fixed acceptance ε =", S.EPS_FIXED, "rad (2 mrad)")
print("thresholds:  γ=1 -> τ=%.3f   γ=2 -> τ=%.3f   γ=3 -> τ=%.3f"
      % (S.threshold(1), S.threshold(2), S.threshold(3)))

import pickle
import qtrk_pipeline as qp
from scipy.sparse.linalg import eigsh
OUT = Path(S.__file__).resolve().parent / "outputs" / "hamiltonian_spectrum"
OUT.mkdir(parents=True, exist_ok=True)
GAMMA = 3.0
SPEC_N = [10, 20, 50, 100, 200, 400, 700, 1000]
N_REPS = {10: 3, 20: 3, 50: 3, 100: 3, 200: 3, 400: 2, 700: 1, 1000: 1}
TAU = S.threshold(GAMMA)
print("spectral grid:", SPEC_N, " τ =", TAU)

fixed acceptance ε = 0.002 rad (2 mrad)
thresholds:  γ=1 -> τ=0.600   γ=2 -> τ=0.433   γ=3 -> τ=0.350


spectral grid: [10, 20, 50, 100, 200, 400, 700, 1000]  τ = 0.35


In [2]:
def one_spec(row):
    ham = S.build_A(row, gamma=GAMMA)
    A = ham.A
    diag = np.asarray(A.diagonal(), float)
    A_abs = abs(A.tocsr())
    off = np.asarray(A_abs.sum(axis=1)).ravel() - np.abs(diag)
    gmin = float((diag - off).min())
    def extreme(which):
        try:
            return float(eigsh(A, k=1, which=which, return_eigenvectors=False,
                               maxiter=4000, tol=1e-6)[0])
        except Exception as e:
            print(f"    eigsh {which} failed n={row['n_trk']}: {type(e).__name__}")
            return float("nan")
    lam_min, lam_max = extreme("SA"), extreme("LA")
    cond = (lam_max / lam_min) if (lam_min == lam_min and lam_min > 0) else float("nan")
    sol = np.asarray(qp.load_solution(row["sol_key"])["sol"], float)
    truth = np.asarray(qp.truth_from_event(S._event_of(row)), bool)
    return dict(n_trk=int(row["n_trk"]), n_seg=int(A.shape[0]), nnz=int(A.nnz),
                lam_min=lam_min, lam_max=lam_max, cond=cond, gersh_min=gmin,
                max_off=float(off.max()), mean_off=float(off.mean()),
                sol_true=sol[truth].copy(), sol_false=sol[~truth].copy())

cache = OUT / "spectral_cache.pkl"
if cache.exists():
    spec = pickle.load(open(cache, "rb"))
    print("loaded spectral cache:", sorted(spec))
else:
    spec = {}
ci = S.solves_index("classical", gamma=GAMMA, hit_ineff=0.0)
for n in SPEC_N:
    if n in spec:
        continue
    rows = ci[ci.n_trk == n].head(N_REPS[n])
    if not len(rows):
        print(f"  n={n}: no fixed-ε γ=3 solves — skipped"); continue
    runs = []
    for _, r in rows.iterrows():
        rec = one_spec(r)
        runs.append(rec)
        print(f"  n={n:4d} n_seg={rec['n_seg']:8d} nnz/n={rec['nnz']/rec['n_seg']:.2f} "
              f"λmin={rec['lam_min']:.3f} λmax={rec['lam_max']:.3f} κ={rec['cond']:.2f} "
              f"gersh_min={rec['gersh_min']:.2f}")
    spec[n] = runs
    pickle.dump(spec, open(cache, "wb"))
print("spectral done:", sorted(spec))

  n=  10 n_seg=     400 nnz/n=1.15 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  10 n_seg=     400 nnz/n=1.15 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  10 n_seg=     400 nnz/n=1.15 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  20 n_seg=    1600 nnz/n=1.07 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  20 n_seg=    1600 nnz/n=1.07 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  20 n_seg=    1600 nnz/n=1.07 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  50 n_seg=   10000 nnz/n=1.03 λmin=2.152 λmax=5.848 κ=2.72 gersh_min=1.00
  n=  50 n_seg=   10000 nnz/n=1.03 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00
  n=  50 n_seg=   10000 nnz/n=1.03 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00


  n= 100 n_seg=   40000 nnz/n=1.02 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00


  n= 100 n_seg=   40000 nnz/n=1.02 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00


  n= 100 n_seg=   40000 nnz/n=1.02 λmin=2.152 λmax=5.848 κ=2.72 gersh_min=1.00


  n= 200 n_seg=  160000 nnz/n=1.01 λmin=2.382 λmax=5.618 κ=2.36 gersh_min=2.00


...

  n= 200 n_seg=  160000 nnz/n=1.01 λmin=0.764 λmax=7.236 κ=9.47 gersh_min=0.00


  n= 400 n_seg=  640000 nnz/n=1.01 λmin=1.201 λmax=6.799 κ=5.66 gersh_min=0.00


  n= 400 n_seg=  640000 nnz/n=1.01 λmin=1.352 λmax=6.648 κ=4.92 gersh_min=0.00


  n= 700 n_seg= 1960000 nnz/n=1.01 λmin=0.764 λmax=7.236 κ=9.47 gersh_min=0.00


  n=1000 n_seg= 4000000 nnz/n=1.01 λmin=1.387 λmax=6.613 κ=4.77 gersh_min=0.00
spectral done: [10, 20, 50, 100, 200, 400, 700, 1000]


## (a,b) Spectrum vs multiplicity

In [3]:
ns = np.array(sorted(spec), float)
def stk(k, red=np.mean):
    return np.array([red([r[k] for r in spec[int(n)]]) for n in ns])
lmin, lmin_s = stk("lam_min"), stk("lam_min", np.std)
lmax, lmax_s = stk("lam_max"), stk("lam_max", np.std)
cond, cond_s = stk("cond"), stk("cond", np.std)
gmin, max_off, n_seg = stk("gersh_min"), stk("max_off"), stk("n_seg")

fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].errorbar(ns, lmin, yerr=lmin_s, fmt="o-", color="#1b7837", capsize=3,
               label=r"$\lambda_{\min}$")
ax[0].errorbar(ns, lmax, yerr=lmax_s, fmt="s-", color="#c51b7d", capsize=3,
               label=r"$\lambda_{\max}$")
ax[0].plot(ns, gmin, "v--", color="grey", label="Gershgorin min$_i(d_i-r_i)$")
ax[0].axhline(0, color="k", lw=0.7, ls=":")
ax[0].axhline(S.DELTA + GAMMA, color="k", lw=0.7, ls=":")
ax[0].text(ns[0], S.DELTA + GAMMA + 0.1, fr"$\delta+\gamma={S.DELTA+GAMMA:g}$",
           fontsize=9, color="grey")
ax[0].set_xscale("log"); ax[0].set_xlabel("n_tracks"); ax[0].set_ylabel("eigenvalue")
ax[0].set_title("(a) Spectral extremes", fontweight="bold"); ax[0].legend(fontsize=9)

ax[1].errorbar(ns, cond, yerr=cond_s, fmt="o-", color="#2166ac", capsize=3,
               label=r"$\kappa=\lambda_{\max}/\lambda_{\min}$")
ax[1].set_xscale("log"); ax[1].set_xlabel("n_tracks"); ax[1].set_ylabel(r"$\kappa$")
ax[1].set_title("(b) Condition number", fontweight="bold"); ax[1].legend(fontsize=9)

ax[2].plot(ns, max_off, "o-", color="#d6604d", label="max off-diag row-sum")
ax[2].plot(ns, stk("mean_off"), "s--", color="#d6604d", alpha=0.6,
           label="mean off-diag row-sum")
ax[2].set_xscale("log"); ax[2].set_xlabel("n_tracks"); ax[2].set_ylabel("off-diag row-sum")
ax2 = ax[2].twinx()
ax2.plot(ns, n_seg, "^-", color="#6a3d9a"); ax2.set_yscale("log")
ax2.set_ylabel("n_segments", color="#6a3d9a"); ax2.tick_params(axis="y", labelcolor="#6a3d9a")
ax[2].set_title("(c) Sparsity & coupling", fontweight="bold"); ax[2].legend(fontsize=9, loc="upper left")
fig.suptitle(fr"Hamiltonian spectrum vs multiplicity (ε=2 mrad, γ={GAMMA:g}, δ={S.DELTA:g})",
             fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"spectrum.{ext}", dpi=dpi, bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n{'n':>5} {'n_seg':>9} {'λmin':>8} {'λmax':>7} {'κ':>8} {'gersh':>8} {'max_off':>8}")
for i, n in enumerate(ns):
    print(f"{int(n):5d} {int(n_seg[i]):9d} {lmin[i]:8.3f} {lmax[i]:7.3f} "
          f"{cond[i]:8.2f} {gmin[i]:8.2f} {max_off[i]:8.1f}")


    n     n_seg     λmin    λmax        κ    gersh  max_off
   10       400    2.382   5.618     2.36     2.00      2.0
   20      1600    2.382   5.618     2.36     2.00      2.0
   50     10000    2.305   5.695     2.48     1.67      2.3
  100     40000    2.305   5.695     2.48     1.67      2.3
  200    160000    1.843   6.157     4.73     1.33      2.7
  400    640000    1.277   6.723     5.29     0.00      4.0
  700   1960000    0.764   7.236     9.47     0.00      4.0
 1000   4000000    1.387   6.613     4.77     0.00      4.0


## (c) True vs false separation as multiplicity grows

In [4]:
hist_n = [n for n in [10, 100, 400, 700, 1000] if n in spec]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
bins = np.linspace(0.0, 1.5, 81)
for ax, n in zip(axes.ravel(), hist_n):
    st = np.concatenate([r["sol_true"] for r in spec[n]])
    sf = np.concatenate([r["sol_false"] for r in spec[n]])
    ax.hist(sf, bins=bins, color="#c51b7d", alpha=0.7, density=True,
            label=f"false ({len(sf):,})")
    ax.hist(st, bins=bins, color="#1b7837", alpha=0.7, density=True,
            label=f"true ({len(st):,})")
    ax.axvline(TAU, color="k", ls="--", lw=1.2, label=f"τ={TAU:.2f}")
    fp = int((sf > TAU).sum())
    ax.set_title(f"n={n}   false>τ: {fp:,} ({fp/max(len(sf),1)*100:.2f}%)",
                 fontsize=10, fontweight="bold")
    ax.set_yscale("log"); ax.set_ylim(1e-2, None)
    ax.set_xlabel("solver output  $s_i$"); ax.set_ylabel("density")
    ax.legend(fontsize=7, loc="upper right")
for ax in axes.ravel()[len(hist_n):]:
    ax.axis("off")
fig.suptitle("Solution distribution by truth vs multiplicity "
             "(separation stays clean iff false peak stays left of τ)",
             fontsize=13, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.96])
for ext, dpi in (("pdf", 600), ("png", 300)):
    fig.savefig(OUT / f"separation_vs_multiplicity.{ext}", dpi=dpi,
                bbox_inches="tight", facecolor="white")
plt.show()
print(f"\n{'n':>5} {'min(true)':>10} {'max(false)':>11} {'gap':>8} {'FP%':>7}")
for n in hist_n:
    st = np.concatenate([r["sol_true"] for r in spec[n]])
    sf = np.concatenate([r["sol_false"] for r in spec[n]])
    print(f"{n:5d} {st.min():10.3f} {sf.max():11.3f} {st.min()-sf.max():+8.3f} "
          f"{(sf>TAU).sum()/max(len(sf),1)*100:7.2f}")


    n  min(true)  max(false)      gap     FP%
   10      0.364       0.250   +0.114    0.00
  100      0.364       0.392   -0.028    0.00
  400      0.364       0.869   -0.505    0.01
  700      0.364       1.500   -1.136    0.01
 1000      0.364       0.938   -0.574    0.03
